# 19 — PHBench external stress (optional)

**Objective.** Fail gracefully when gated data are unavailable; otherwise estimate clean-to-retrospective/leaky feature curves without calling PHBench an independent label-multiverse replication.

**Scientific contract.** This notebook reports no empirical result until it executes successfully against hash-verified inputs. It writes immutable outputs plus a completion manifest. Expected counts are protocol assertions, not substituted observations.

No data-access control is bypassed. Follower and retrospective engagement variables are explicitly labelled leakage stressors.

In [ ]:
# Standard CRUX-VC Colab bootstrap. Git dotfiles restore from the Drive project root; tokens never appear in cells.
import os, subprocess, sys
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass

import shutil
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/CRUX_Research")
for _dotfile in (".gitconfig", ".git-credentials"):
    if (DRIVE_PROJECT_ROOT / _dotfile).exists():
        shutil.copy(DRIVE_PROJECT_ROOT / _dotfile, Path.home() / _dotfile)
if (Path.home() / ".git-credentials").exists():
    os.chmod(Path.home() / ".git-credentials", 0o600)

REPO_URL = "https://github.com/anasbiswas1/crux-vc"
REPO_ROOT = Path(os.environ.get("CRUX_REPO_ROOT", "/content/drive/MyDrive/CRUX_Research/crux-vc"))
if not (REPO_ROOT / ".cruxvc-root").exists():
    if REPO_ROOT.exists() and any(REPO_ROOT.iterdir()):
        raise RuntimeError(f"{REPO_ROOT} exists but is not a CRUX-VC checkout")
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    import yaml, pandas, sklearn, pyarrow  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")], check=True)

from cruxvc.runtime import bootstrap_notebook
CTX = bootstrap_notebook("19", suffix=None)
P, CFG, PROFILE = CTX.paths, CTX.config, CTX.profile

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from cruxvc.external import (
    locate_kaggle_snapshot_data,
    locate_phbench_data,
    read_phbench_tables,
    read_snapshot_table,
)
from cruxvc.io import write_json, write_table


def make_binary_pipeline(train, columns):
    numeric = [c for c in columns if pd.api.types.is_numeric_dtype(train[c])]
    categorical = [c for c in columns if c not in numeric and train[c].nunique(dropna=True) <= 250]
    usable = numeric + categorical
    if not usable:
        raise RuntimeError("No usable numeric or low-cardinality categorical features were detected")
    transformers = []
    if numeric:
        transformers.append(("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric))
    if categorical:
        try:
            encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
        except TypeError:
            encoder = OneHotEncoder(handle_unknown="ignore", sparse=True)
        transformers.append(("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", encoder)]), categorical))
    model = Pipeline([
        ("preprocess", ColumnTransformer(transformers)),
        ("model", LogisticRegression(max_iter=3000, class_weight=None)),
    ])
    return model, usable


def binary_metrics(y, probability):
    prevalence = float(np.mean(y))
    ap = float(average_precision_score(y, probability))
    return {
        "average_precision": ap,
        "ap_lift": ap / prevalence if prevalence else np.nan,
        "log_loss": float(log_loss(y, probability, labels=[0, 1])),
        "prevalence": prevalence,
    }

phbench_status_path = P.audits / "19_phbench_status.json"
phbench_curve_path = P.inference / "phbench_clean_to_leaky_curve.csv"
kaggle_status_path = P.audits / "19_kaggle_snapshot_status.json"
kaggle_curve_path = P.inference / "kaggle_snapshot_leakage_negative_control.csv"


In [ ]:
phbench_availability = locate_phbench_data()
if not phbench_availability["available"]:
    phbench_curve = pd.DataFrame(columns=["feature_set", "average_precision", "ap_lift", "log_loss", "status"])
    write_json({
        **phbench_availability,
        "status": "optional_skipped_without_failure",
        "scientific_consequence": "No external PHBench stress claim may be made.",
    }, phbench_status_path)
else:
    tables = read_phbench_tables(phbench_availability["path"])
    data = pd.concat([frame.assign(_source_table=name) for name, frame in tables.items()], ignore_index=True, sort=False)
    label_candidates = [c for c in data.columns if c.lower() in {"label", "target", "series_a", "series_a_18m", "success"}]
    if not label_candidates:
        raise RuntimeError(f"PHBench label could not be identified from columns: {list(data.columns)}")
    label = label_candidates[0]
    data = data[data[label].notna()].copy()
    data[label] = pd.to_numeric(data[label], errors="raise").astype(int)
    for column in [c for c in data.columns if any(token in c.lower() for token in ["title", "description", "text"])] :
        values = data[column].fillna("").astype(str)
        data[f"{column}__character_count"] = values.str.len()
        data[f"{column}__token_count"] = values.str.split().str.len()
    date_candidates = [c for c in data.columns if "date" in c.lower() or "time" in c.lower()]
    if date_candidates:
        data["_order"] = pd.to_datetime(data[date_candidates[0]], errors="coerce")
        data = data.sort_values("_order")
    else:
        data = data.sort_index()
    split = int(len(data) * 0.8)
    train, validation = data.iloc[:split].copy(), data.iloc[split:].copy()
    excluded = {label, "_source_table", "_order"}
    candidates_all = [c for c in data.columns if c not in excluded and not any(token in c.lower() for token in ["url", "name", "id"])]
    text_shape = [c for c in candidates_all if c.endswith("__character_count") or c.endswith("__token_count")]
    topic_calendar = [c for c in candidates_all if any(token in c.lower() for token in ["topic", "category", "launch", "calendar", "weekday", "hour", "month"])]
    clean = list(dict.fromkeys(text_shape + topic_calendar))
    rank = clean + [c for c in candidates_all if "rank" in c.lower()]
    engagement = rank + [c for c in candidates_all if any(token in c.lower() for token in ["vote", "comment", "engagement"])]
    follower = engagement + [c for c in candidates_all if "follower" in c.lower()]
    feature_sets = {
        "static_clean": list(dict.fromkeys(clean)),
        "add_rank": list(dict.fromkeys(rank)),
        "add_retrospective_engagement": list(dict.fromkeys(engagement)),
        "add_explicit_follower_leakage": list(dict.fromkeys(follower)),
    }
    rows = []
    for name, columns in feature_sets.items():
        columns = [c for c in columns if c in data.columns]
        if not columns:
            rows.append({"feature_set": name, "status": "no_detected_columns"})
            continue
        model, usable = make_binary_pipeline(train, columns)
        model.fit(train[usable], train[label])
        probability = model.predict_proba(validation[usable])[:, 1]
        rows.append({
            "feature_set": name,
            "status": "executed",
            "n_features_raw": len(usable),
            "validation_n": len(validation),
            "positives": int(validation[label].sum()),
            **binary_metrics(validation[label], probability),
        })
    phbench_curve = pd.DataFrame(rows)
    write_json({
        **phbench_availability,
        "status": "executed",
        "label_column": label,
        "role": "optional_one_outcome_leakage_stress_not_rq1_replication",
        "limitations": ["one outcome only", "access/timing caveats remain", "not a label-multiverse replication"],
    }, phbench_status_path)
write_table(phbench_curve, phbench_curve_path)


In [ ]:
kaggle_availability = locate_kaggle_snapshot_data()
if not kaggle_availability["available"]:
    kaggle_curve = pd.DataFrame(columns=["feature_set", "average_precision", "ap_lift", "log_loss", "status"])
    write_json({
        **kaggle_availability,
        "status": "optional_skipped_without_failure",
        "scientific_consequence": "No leaky-snapshot negative-control result may be reported.",
    }, kaggle_status_path)
else:
    snapshot, selected_file = read_snapshot_table(kaggle_availability["path"])
    normalized = {str(column).lower(): column for column in snapshot.columns}
    target_column = next((normalized[name] for name in ["label", "target", "success", "is_success"] if name in normalized), None)
    target_definition = None
    if target_column is not None:
        y = pd.to_numeric(snapshot[target_column], errors="coerce")
        if set(y.dropna().unique()) - {0, 1}:
            raise RuntimeError(f"Detected target {target_column} is not binary")
        target_definition = f"provided_binary_column:{target_column}"
    elif "status" in normalized:
        target_column = normalized["status"]
        status = snapshot[target_column].astype("string").str.lower().str.strip()
        recognized = status.isin(["acquired", "ipo", "operating", "closed"])
        snapshot = snapshot[recognized].copy()
        status = snapshot[target_column].astype("string").str.lower().str.strip()
        y = status.isin(["acquired", "ipo"]).astype(int)
        target_definition = "snapshot_status_acquired_or_ipo_vs_operating_or_closed"
    else:
        raise RuntimeError("Could not identify a binary label or snapshot status in the Kaggle negative-control table")
    snapshot = snapshot.loc[y.notna()].copy()
    y = y.loc[y.notna()].astype(int)
    snapshot["_negative_control_target"] = y.to_numpy()
    if snapshot["_negative_control_target"].value_counts().min() < 20:
        raise RuntimeError("Kaggle negative-control target has fewer than 20 cases in one class")
    identifier_tokens = ["name", "permalink", "homepage", "url", "uuid", "id"]
    leakage_tokens = ["funding_total", "funding_round", "last_funding", "last_milestone", "raised_amount", "acquisition", "ipo", "status"]
    all_candidates = [
        c for c in snapshot.columns
        if c not in {target_column, "_negative_control_target"}
        and not any(token in c.lower() for token in identifier_tokens)
    ]
    clean_columns = [c for c in all_candidates if not any(token in c.lower() for token in leakage_tokens)]
    leaky_columns = list(dict.fromkeys(clean_columns + [c for c in all_candidates if any(token in c.lower() for token in leakage_tokens)]))
    train, validation = train_test_split(
        snapshot,
        test_size=0.20,
        stratify=snapshot["_negative_control_target"],
        random_state=int(CFG["execution"]["random_seed"]),
    )
    rows = []
    for name, columns in {"snapshot_metadata_only": clean_columns, "add_cumulative_outcome_accumulating_fields": leaky_columns}.items():
        model, usable = make_binary_pipeline(train, columns)
        model.fit(train[usable], train["_negative_control_target"])
        probability = model.predict_proba(validation[usable])[:, 1]
        rows.append({
            "feature_set": name,
            "status": "executed",
            "role": "leaky_snapshot_negative_control_only",
            "n_features_raw": len(usable),
            "validation_n": len(validation),
            "positives": int(validation["_negative_control_target"].sum()),
            **binary_metrics(validation["_negative_control_target"], probability),
        })
    kaggle_curve = pd.DataFrame(rows)
    write_json({
        **kaggle_availability,
        "selected_file": selected_file,
        "status": "executed",
        "target_definition": target_definition,
        "role": "leaky_snapshot_negative_control_only_not_replication",
        "random_split": True,
    }, kaggle_status_path)
write_table(kaggle_curve, kaggle_curve_path)
CTX.recorder.complete(
    [phbench_status_path, phbench_curve_path, kaggle_status_path, kaggle_curve_path],
    extra={
        "phbench_optional_skipped": not phbench_availability["available"],
        "kaggle_optional_skipped": not kaggle_availability["available"],
    },
)
print("PHBench status:", "executed" if phbench_availability["available"] else "skipped")
print("Kaggle snapshot status:", "executed" if kaggle_availability["available"] else "skipped")
